# Notebook 03: tiny scalar backward engine

In notebook 02, we computed the chain rule by hand. In this notebook, we build a tiny teaching engine that stores the same forward values and backward sensitivities inside Python objects. The goal is not to replace the production MLP code; it is to make the idea behind `backward()` visible.

## Start with the manual expression

First, repeat the same expression from notebook 02. These plain floats give us a baseline: we know the forward values should be `d = -6.0`, `e = 4.0`, and `f = 16.0`. Later, our `Value` objects should produce the same forward numbers while also remembering the graph.

In [101]:
a = 2.0
b = -3.0
c = 10.0

d = a * b
e = d + c
f = e * e

print(d, e, f)

-6.0 4.0 16.0


## From hand bookkeeping to object bookkeeping

The manual backward pass worked because we kept track of each intermediate value and each local derivative. A `Value` object will keep that bookkeeping next to the number itself. For now, it stores the forward number in `data`, the backward sensitivity in `grad`, the parent values in `_prev`, and the operation name in `_op`.

## Add local backward rules and `.backward()` to `Value`

Each operation also stores a small `_backward` function on its result. That function knows how to pass `grad` back to the `Value` objects that made the result. The `backward()` method starts from the final result, sets its `grad` to `1.0`, and runs those small functions from the end back to the start.

In [102]:
import os
from typing import Callable


class Value:
    """A number that also remembers how it was made.

    data is the number.
    grad will later hold how much the final answer changes because of this number.
    """

    def __init__(
        self,
        data: float,
        _children: tuple["Value", ...] = (),
        _op: str = "",
    ) -> None:
        """Store the number, its starting grad, and the Values that made it."""
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self._backward: Callable[[], None] = lambda: None

    def __repr__(self) -> str:
        """Make print(value) easy to read."""
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other: "Value") -> "Value":
        """Make a new Value for self + other."""
        out = Value(data=self.data + other.data, _children=(self, other), _op="+")

        def _backward() -> None:
            """Move this result's grad back to both inputs."""
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other: "Value") -> "Value":
        """Make a new Value for self * other."""
        out = Value(data=self.data * other.data, _children=(self, other), _op="*")

        def _backward() -> None:
            """Move this result's grad back using the other input's number."""
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, power:int) -> "Value":
        """Make a new Value for self ** 2."""
        if power != 2:
            raise ValueError(f"Only powers of 2 are supported, not {power}.")

        out = Value(
            data=self.data * self.data,
            _children=(self,),
            _op=f"**{power}",
        )

        def _backward() -> None:
            """Move this result's grad back using the other input's number.

            backward rule: near x = 3.0, changing x changes y about 2 * x times as much.
            """
            self.grad += 2 * self.data * out.grad

        out._backward = _backward
        return out

    def backward(self) -> None:
        """Fill in grads, starting from this final Value.

        First find all earlier Values.
        Then work backward so each Value passes its grad to the Values that made it.
        """
        topo: list[Value] = []
        visited: set[Value] = set()

        def _build_topo(value: Value) -> None:
            """Put earlier Values before later Values."""
            if value in visited:
                return

            visited.add(value)

            for child in value._prev:
                _build_topo(child)

            topo.append(value)


        _build_topo(self)

        if os.environ.get("DEBUG"):
            print(f"Order: {topo}")

        self.grad = 1.0

        for node in reversed(topo):
            node._backward()

## Leaf values

A leaf value is an input we create directly, like `a = Value(2.0)`. It has a number in `data`, starts with `grad = 0.0`, and has no parents because no earlier operation created it.

In [103]:
a = Value(2.0)
b = Value(-3.0)

print(a)
print(b)

Value(data=2.0, grad=0.0)
Value(data=-3.0, grad=0.0)


## Addition creates a graph node

When we write `c = a + b`, Python calls `a.__add__(b)`. The result `c` stores the forward value `-1.0`, remembers that the operation was `+`, and keeps links back to both parent objects. Those links are what the backward pass will use later to update `a.grad` and `b.grad`.

In [104]:
a = Value(2.0)
b = Value(-3.0)
c = a + b

print(c)
print(c._op)
print(c._prev)

Value(data=-1.0, grad=0.0)
+
{Value(data=-3.0, grad=0.0), Value(data=2.0, grad=0.0)}


## Inspect the parent links

The parent links store the actual `Value` objects, not just the numbers `2.0` and `-3.0`. Printing only each parent's `.data` is a gentle way to check which forward values are connected to `c`. Because `_prev` is a set, the order may change, but both parents should be present.

In [105]:
print(c.data)
print([parent.data for parent in c._prev])

-1.0
[-3.0, 2.0]


## Rebuild the expression with `Value` objects

Now use the same expression shape as the plain-float example, but stop at `e = d + c`. The forward numbers should still match notebook 02. The difference is that `d` and `e` now also carry parent links and local backward rules.

In [106]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

d = a * b
e = d + c

print(d)
print(e)
print(e._op)
print([parent.data for parent in e._prev])

Value(data=-6.0, grad=0.0)
Value(data=4.0, grad=0.0)
+
[10.0, -6.0]


## Finish the forward graph

The final step is `f = e * e`, so the same `Value` object is used twice. The printed parent list only shows one `4.0` because `_prev` is a set, and a set keeps one copy of the same object. During backward, this repeated use is still important: `e` should receive gradient from both sides of the multiplication.

In [107]:
f = e * e

print(f)
print(f._op)
print([parent.data for parent in f._prev])

Value(data=16.0, grad=0.0)
*
[4.0]


## Test one local backward rule

Before automating the whole graph, test one operation by hand. In this tiny test, we pretend `c` is the final output. A value's gradient means "how much the chosen final output changes when this value changes."

That is why we set `c.grad = 1.0`: if `c` is the chosen final output, then `dc/dc = 1.0`. In plain words, `c` changes one-for-one with itself. Calling `c._backward()` then applies only the local addition rule, so both `a.grad` and `b.grad` should increase by `1.0`.

In [108]:
a = Value(2.0)
b = Value(-3.0)
c = a + b

c.grad = 1.0
c._backward()

print(a.grad)
print(b.grad)

1.0
1.0


## Test the local multiplication rule

Now test one multiplication node by pretending its output is the final value. Setting `c.grad = 1.0` means `dc/dc = 1.0`. For `c = a * b`, the gradient sent to `a` is `b.data * c.grad`, and the gradient sent to `b` is `a.data * c.grad`.

In [109]:
a = Value(2.0)
b = Value(-3.0)
c = a * b

c.grad = 1.0
c._backward()

print(a.grad)
print(b.grad)

-3.0
2.0


## Test the local pow rule

In [110]:
x = Value(3.0)
y = x ** 2

y.backward()

print(y.data)
print(x.grad)

Order: [Value(data=3.0, grad=0.0), Value(data=9.0, grad=0.0)]
9.0
6.0


## Trace `a.grad` through the full expression

Now connect the local rules by hand. The full expression is `d = a * b`, `e = d + c`, and `f = e * e`. Setting `f.grad = 1.0` only seeds the backward pass; it does not update `a`, `b`, or `c` until we call the stored `_backward()` functions.

The order matters. First, `f._backward()` uses the saved history from `f = e * e` and gives `e.grad = 8.0`. Next, `e._backward()` uses the saved history from `e = d + c` and gives `d.grad = 8.0` and `c.grad = 8.0`. Finally, `d._backward()` uses the saved history from `d = a * b`, so `a.grad = b.data * d.grad = -3.0 * 8.0 = -24.0`.

In [111]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

d = a * b
e = d + c
f = e * e

f.grad = 1.0

f._backward()  # sends gradient from f to e
e._backward()  # sends gradient from e to d and c
d._backward()  # sends gradient from d to a and b

print(f"f.grad = {f.grad} with parents: {f._prev}")
print(f"e.grad = {e.grad} with parents: {e._prev}")
print(f"d.grad = {d.grad} with parents: {d._prev}")
print(a.grad)
print(b.grad)
print(c.grad)


f.grad = 1.0 with parents: {Value(data=4.0, grad=8.0)}
e.grad = 8.0 with parents: {Value(data=10.0, grad=8.0), Value(data=-6.0, grad=8.0)}
d.grad = 8.0 with parents: {Value(data=-3.0, grad=16.0), Value(data=2.0, grad=-24.0)}
-24.0
16.0
8.0


## Step-by-step order inside `f.backward()`

The previous section called `f._backward()`, `e._backward()`, and `d._backward()` by hand. Now `f.backward()` does those steps for us. Inside this call, `self` is `f`, because `f` is the object we call the method on.

First, `backward()` builds a list named `topo`. This list puts earlier values before later values. For this expression, one possible order is:

```text
a, b, d, c, e, f
```

The exact order of siblings can change because `_prev` is a set. The important part is:

```text
a and b come before d
d and c come before e
e comes before f
```

The calls that build the list happen like this:

```text
_build_topo(f)
  _build_topo(e)
    _build_topo(d)
      _build_topo(a) -> add a to topo
      _build_topo(b) -> add b to topo
    add d to topo
    _build_topo(c) -> add c to topo
  add e to topo
add f to topo
```

Then `backward()` sets the final value's gradient:

```python
f.grad = 1.0
```

Finally, it walks through `topo` backward. That means it runs the stored functions in this kind of order:

```text
f._backward()  -> gives grad to e
e._backward()  -> gives grad to d and c
d._backward()  -> gives grad to a and b
c._backward()  -> nothing to do
b._backward()  -> nothing to do
a._backward()  -> nothing to do
```

That is why one call to `f.backward()` can fill in all the grads. Rebuild the expression so every gradient starts at `0.0`, then call `f.backward()` once.

In [112]:
os.environ["DEBUG"] = "1"
a = Value(2.0)  # rebuild the expression so gradients start at 0
b = Value(-3.0)
c = Value(10.0)

d = a * b
e = d + c
f = e * e

f.backward()

print(f"f.grad = {f.grad}")
print(f"e.grad = {e.grad}")
print(f"d.grad = {d.grad}")
print(f"a.grad = {a.grad}")
print(f"b.grad = {b.grad}")
print(f"c.grad = {c.grad}")
os.environ["DEBUG"] = "0"

Order: [Value(data=10.0, grad=0.0), Value(data=-3.0, grad=0.0), Value(data=2.0, grad=0.0), Value(data=-6.0, grad=0.0), Value(data=4.0, grad=0.0), Value(data=16.0, grad=0.0)]
f.grad = 1.0
e.grad = 8.0
d.grad = 8.0
a.grad = -24.0
b.grad = 16.0
c.grad = 8.0


In [113]:
expected_grads = {
       "a": -24.0,
       "b": 16.0,
       "c": 8.0,
}

actual_grads = {
    "a": a.grad,
    "b": b.grad,
    "c": c.grad,
}

for name in expected_grads:
    print(f"{name}.grad from backward = {actual_grads[name]}")
    print(f"{name}.grad expected      = {expected_grads[name]}")
    print()

a.grad from backward = -24.0
a.grad expected      = -24.0

b.grad from backward = 16.0
b.grad expected      = 16.0

c.grad from backward = 8.0
c.grad expected      = 8.0



In [114]:
print(f"a.grad from backward = {a.grad}")
print("a.grad from notebook 02 = -24.0")

print(f"b.grad from backward = {b.grad}")
print("b.grad from notebook 02 = 16.0")

print(f"c.grad from backward = {c.grad}")
print("c.grad from notebook 02 = 8.0")

a.grad from backward = -24.0
a.grad from notebook 02 = -24.0
b.grad from backward = 16.0
b.grad from notebook 02 = 16.0
c.grad from backward = 8.0
c.grad from notebook 02 = 8.0


## Connection to PyTorch

PyTorch does this same kind of bookkeeping for tensors. During the forward pass,
it records which operations created each result. When you call `.backward()` on
a final value, PyTorch walks that recorded history backward and fills in
gradients.

Our `Value` class is a tiny teaching version of that idea. It only works with
single numbers and a few operations, but the main pattern is the same: store the
forward value, remember how it was made, then run backward to compute grads.